# Task 9 — Smart Electricity Consumption Optimizer
## One notebook with an interactive dashboard

**Run all cells, then use the dashboard at the bottom.** Choose a historical
electricity case, enter household settings, and click **Run optimizer**.
The result includes the forecast, recommendations, schedule, estimated job
savings, ontology/rule trace, SHAP explanation and LSTM activation heatmap.

This is a separate, self-contained notebook. Your supplied Protégé ontology
and unchanged Task 7 source are embedded, so no output ZIP transfer is needed.
If this runtime has a completed updated Task 7 folder, its models are reused.
**In a fresh runtime without saved models, the default settings download the
real UCI data and run Task 7 once first.** That can take several minutes; a
Colab GPU helps. The small classical ML stage then trains once and is cached.
Opening a different Colab notebook does not automatically share its runtime.
You can optionally set `TASK7_FOLDER` to a saved folder instead of retraining.

Electricity history is real recorded UCI data, not a live meter feed. Household
occupancy, temperature, tariff and appliance settings are editable scenario
inputs. No real appliance is switched. Savings are estimates for requested
schedulable jobs, not measured reductions in a household bill.

In [ ]:
import importlib.util
import subprocess
import sys
requirements = {"tensorflow": "tensorflow==2.20.0", "shap": "shap>=0.48,<0.51",
    "ipywidgets": "ipywidgets>=8,<9", "numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.2,<3",
    "sklearn": "scikit-learn>=1.5,<2", "matplotlib": "matplotlib>=3.8,<4", "joblib": "joblib>=1.4,<2"}
missing = [spec for name, spec in requirements.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if "google.colab" in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()

## How the previous tasks connect
| Component | Earlier tasks | Role here |
|---|---|---|
| Ontology and inference | 1–2 | Imported OWL/SWRL plus operational decision rules |
| Data and feature preparation | 4–5 | Updated Task 7's shared causal preprocessing |
| Deep learning | 7 | Frozen LSTM extracts temporal patterns |
| Classical ML | 6 | Regressor predicts demand from the learned patterns |
| Scheduling | 3 | A* finds a permitted low-cost job schedule |
| Explanation | 8 | SHAP for the ML stage and LSTM activation visualization |

## 1. Settings
Leave `TASK7_FOLDER` empty for automatic discovery. If no saved Task 7 results
are found, `BUILD_TASK7_IF_MISSING=True` creates them using your current Task 7
implementation. Set it to False if you only want to use existing model files.

Training/validation caps limit the *new ML head's* fitting cost. Test metrics
still use the full saved test set. No test targets are used to select models
or make operational recommendations.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from pathlib import Path
import tempfile
import json
from IPython.display import display, HTML

TASK7_FOLDER = ""  # Optional: a completed task7_FINAL_outputs/full_... directory.
BUILD_TASK7_IF_MISSING = True
TRAIN_EXAMPLES = 20000
VALIDATION_EXAMPLES = 5000
FORCE_REFIT_ML = False
TASK9_OUTPUT_DIR = Path("task9_outputs")
TASK9_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Embedded project resources
The next collapsed cell contains byte-for-byte compressed copies of your
ontology and Task 7 source. SHA-256 checks verify them before use. Nothing in
the ontology is treated as an instruction to execute Python.

In [ ]:
import base64, hashlib, zlib
ONTOLOGY_XML = zlib.decompress(base64.b64decode('eNrtXFtz4jYUfs+vcNmX9gHMZbfbMFl2MmHTZDYbGDbtdvqSke0DqLUljySH8O8r2QZsbHJpYssE+wESW5fvnPOdoyNZ4uTzvecad8A4puRTo9NqNz4Pjk6YM+1PhueGfEj4p8ZcCL9vmovFosXBQ0RgewFWi7KZOUMBQ3f2nGIxZ0g+wyYlgrp0hoGb3Xb3V/OjGcgawgWnGT9aNrtm48hQl+yhbyEORXdCeJ8u3FQvi17YeLfd7prtj6Z8/C5VXiohr3zn+PjYlFW63aYs0eRLItB9k/B0ZfmZV/mvb1eqgd9MgjzgPrIhXYs7OyB2VNXv9lxqJoOS76jTNmW1EGNOPb5gu9TRMzsdUz3P1kDrKuq/FheITClzWuAESZP0Wr2wAGpltKpuW491bKXr5Ji2EL68awzCbk8k6v4ovm0oZ0AWDUQhfTbMqM+jo6jrn5pNI/zLLOaK246/jFNCqEBCer/hM+oDExJuukxxMIxmc3CUkf75BMN8ErjwhSBL6nbTaGjGjYDjSL48g/7fvjQbb2T9A7YwxtUy3Kt65BzxUzukZ9qskegPmLQYFHGAWMNwwGdgIyFppyA4SFJt6cMTgrhFqQtINihYACdmurE4DJlZSQdlaX0INuYV0PsKR1Lzaszr88BawRhNQyAMOA2YDY+N8oL6aTFWbrxp3KEewmR3q68m4anvuxgRmQdkMDBEZlAGhIjZZgVI94XcYUaJB0To5l0CSk29oiAklIzcM0ocXBkqXtCAw5y6jm4iroHUNCwKQqjiC6XiKjDvBjE8neqmXYSi5lxREGL9PotwJU4thjKTrdqM8NVT+hULrpEHW/42jDP5MmcYSTiNPfSN/IkOFwyTWZro29otLbaObDvwpXBL/eZeQ3ndGKsk2esIm88iTATMgFWERmO6ADZBQhJbP5ESYGoq7R+VGKYMiwoEpBWSmkT7Nqp9l/0GXD+DIhw1f/aNP9FcpCosSqKpubR3XAJPdihtxyowpUqAqZn0BCY5YGMPuVWgEubnLtxjy9VNow2QmkLPeK9ZCQqNAf17IZFrp9AKSE2h4ihU4uromYs4f4urorl7LUJpi/aR9FvowuVcUVaLqNv+UrS0Xwiw2fKMEh54vi4DZ0CUJ33eS2Y9KnjodXfhepAhS95GrlbyZ1HkjYkhlIcGxOKcMRpuQgDl2GWVXmo1SwbEoVvlAs/mqxVRrZbJBXLw1lnt2NBike3tIkVLOwGbenLUcJCukSuNoCy5c3fClCJvendG4XJiD8YgfVwLnTe9lyXvHxzYmMEUGGgKqmkEmrfxX8pE8A47AXLf4IxyCC5a3ubOK9U+F2cjfNE2TyLZGj/7amVB01brLS2U44FfAfzRdSXMkoJy6HaZwJThmXplQdltR69h0lh0WWbX2mJZAHLmq0kkOVX6ydMzxQNM+49+Ct8EjIzOzysRW9JYDj24/EB8jsnsG7LlF9zeABd6zZMD6FDDTHb9pVpRJpU8PY/KJWbS6miuhhw6JMoQuM1wtLxerN+Y6Hlekney/085wVVHmDe23JJhkCtZot/wSHQ/ffT6ZWdgc1rcXuaKViHEYD0qG2jlMNzgcxq4EgSE7z1H161QqE2lrbZcZIE7yI7vt9GIGleOSr1U3Zeen3JodbNvUWe5ufewul/Y/6mg3hWW4dXc0eQUMy6yz54G6oXgwhm/QpiHLq0yWxWVE3cHq3PS2tO/fJCIzQJFuE7x+GQo2AFnh0Nni+yw/EmMXTxqy50/iEKwu43uAVSR/285RaTQOSCndhQZRdaj7GoXxNO8Jj7HtSzTcZJH9/fDY7Jwurqmb2/FmTeOu2f5xXrvYSK9sBExLDAclQmD86TkIpPV34Z5dJ1a1KnFI1O+OrOoM4s6s6gzi1desjmIxCK6NRmeD47kFa5B/Q5EvciQKYS1NMQcjNGPK+N0fGn8HP/2pfG+9aHVPW5Jfb5vtj80O72bTrff6fTbvb9/CVdjuZRnJm0QWC055qulL5urT+TjaCXoP0rI38s=')).decode("utf-8")
assert hashlib.sha256(ONTOLOGY_XML.encode("utf-8")).hexdigest() == '2ada3ac3ccafb385f087d822ddd6dadd9a16b76bbecffc1c58cddbfa28757320'
TASK7_SOURCE_TEXT = zlib.decompress(base64.b64decode('...')).decode("utf-8")
assert hashlib.sha256(TASK7_SOURCE_TEXT.encode("utf-8")).hexdigest() == 'e4d0a2d6d84cdbc11b6ba71d1733d84fce6747b5357b0955d2f2623d9a2fc960'

## 3. Knowledge representation, inference and scheduling
The adapter reads the actual classes, named individuals, subclass links and
two enabled SWRL rules from your OWL file. It supports this file's unary
ClassAtom → hasDecision IndividualPropertyAtom rules and named subclasses.
It is not a general OWL DL consistency reasoner; unsupported SWRL forms fail
explicitly. Asserted example decisions are recorded separately from inferred
decisions, so an existing assertion is not presented as newly inferred.

SWRL candidates feed ordered operational rules for essential appliances,
occupancy, comfort, priority, tariffs and the forecast. Essential protection
overrides conflicting delay candidates. The A* scheduler uses an admissible
remaining-cost bound and checks runtime, deadlines and parallel-job limits.
It reports infeasibility or search limits rather than inventing a schedule.

In [ ]:
"""Task 9: ontology → LSTM features → ML forecast → rules → A* scheduling.

The ontology adapter executes this project's named-subclass and unary class-to-
decision SWRL subset. It is not a general OWL DL consistency reasoner.
"""
import hashlib
import heapq
import json
import math
from dataclasses import asdict, dataclass, replace
from itertools import count
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def local_name(iri):
    return str(iri).rsplit("}", 1)[-1].rsplit("#", 1)[-1].rsplit("/", 1)[-1]


class OntologyKnowledge:
    NS = {"rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
          "rdfs": "http://www.w3.org/2000/01/rdf-schema#", "owl": "http://www.w3.org/2002/07/owl#",
          "swrl": "http://www.w3.org/2003/11/swrl#", "swrla": "http://swrl.stanford.edu/ontologies/3.3/swrla.owl#"}

    def __init__(self, xml_text):
        self.digest = hashlib.sha256(xml_text.encode("utf-8")).hexdigest()
        root = ET.fromstring(xml_text)
        resource = "{" + self.NS["rdf"] + "}resource"
        about = "{" + self.NS["rdf"] + "}about"
        self.parents, self.individual_types, self.asserted_decisions = {}, {}, {}
        self.classes = set()
        for element in root.findall("owl:Class", self.NS):
            name = local_name(element.attrib[about])
            self.classes.add(name)
            self.parents[name] = {local_name(n.attrib[resource]) for n in element.findall("rdfs:subClassOf", self.NS)
                                  if resource in n.attrib}
        for element in root.findall("owl:NamedIndividual", self.NS):
            name = local_name(element.attrib[about])
            self.individual_types[name] = {local_name(n.attrib[resource]) for n in element.findall("rdf:type", self.NS)}
            self.asserted_decisions[name] = {local_name(n.attrib[resource]) for n in element
                                             if local_name(n.tag) == "hasDecision" and resource in n.attrib}
        self.rules = []
        for element in root:
            types = {n.attrib.get(resource) for n in element.findall("rdf:type", self.NS)}
            if self.NS["swrl"] + "Imp" not in types and element.tag != "{" + self.NS["swrl"] + "}Imp":
                continue
            enabled = element.find("swrla:isRuleEnabled", self.NS)
            if enabled is not None and (enabled.text or "").strip().lower() == "false":
                continue
            label_node = element.find("rdfs:label", self.NS)
            label = label_node.text if label_node is not None else f"SWRL_{len(self.rules)+1}"
            body = self._atoms(element.find("swrl:body", self.NS))
            head = self._atoms(element.find("swrl:head", self.NS))
            if not body or not head or any(a["kind"] != "ClassAtom" for a in body):
                raise ValueError(f"Unsupported SWRL body in {label}; this adapter will not silently ignore it.")
            variables = {a["arg1"] for a in body}
            if len(variables) != 1 or any(a["kind"] != "IndividualPropertyAtom" or a["predicate"] != "hasDecision"
                                           or a["arg1"] not in variables for a in head):
                raise ValueError(f"Unsupported SWRL head in {label}.")
            self.rules.append({"id": label, "classes": [a["predicate"] for a in body],
                               "decisions": [a["arg2"] for a in head]})
        if not self.rules:
            raise ValueError("The ontology contains no supported enabled SWRL rules.")

    def _atoms(self, container):
        if container is None or not len(container):
            return []
        resource = "{" + self.NS["rdf"] + "}resource"
        current, result = container[0], []
        while current is not None:
            first = current.find("rdf:first", self.NS)
            if first is None or not len(first):
                raise ValueError("Unsupported SWRL RDF list encoding.")
            atom = first[0]
            kind = atom.find("rdf:type", self.NS)
            kind = local_name(kind.attrib[resource]) if kind is not None else local_name(atom.tag)
            def value(name):
                node = atom.find("swrl:" + name, self.NS)
                return local_name(node.attrib[resource]) if node is not None and resource in node.attrib else None
            result.append({"kind": kind, "predicate": value("classPredicate") or value("propertyPredicate"),
                           "arg1": value("argument1"), "arg2": value("argument2")})
            rest = current.find("rdf:rest", self.NS)
            if rest is None:
                raise ValueError("SWRL list has no terminator.")
            if rest.attrib.get(resource) == self.NS["rdf"] + "nil":
                break
            if not len(rest):
                raise ValueError("Only nested SWRL RDF lists are supported by this adapter.")
            current = rest[0]
        return result

    def infer(self, appliance):
        types = set(self.individual_types.get(appliance.ontology_id, set())) | {"Appliance"}
        if appliance.essential:
            types.add("EssentialAppliance")
        if appliance.flexible:
            types.add("FlexibleAppliance")
        if appliance.priority == "High":
            types.add("HighPriorityAppliance")
        changed = True
        while changed:
            expanded = types | set().union(*(self.parents.get(t, set()) for t in types))
            changed = expanded != types
            types = expanded
        decisions, fired = set(), []
        for rule in self.rules:
            if set(rule["classes"]) <= types:
                decisions.update(rule["decisions"])
                fired.append("OWL:" + rule["id"])
        return {"types": sorted(types), "candidate_decisions": sorted(decisions), "fired": fired,
                "asserted_decisions": sorted(self.asserted_decisions.get(appliance.ontology_id, set()))}


@dataclass(frozen=True)
class Appliance:
    name: str
    power_w: float
    duration_min: int = 60
    requested: bool = True
    on: bool = False
    essential: bool = False
    flexible: bool = True
    priority: str = "Medium"
    earliest_min: int = 0
    deadline_min: int = 360
    requires_occupancy: bool = False
    ontology_id: str = ""


@dataclass(frozen=True)
class Scenario:
    occupancy: int = 3
    temperature_c: float = 35.0
    peak_start_hour: int = 18
    peak_end_hour: int = 20
    peak_rate: float = 9.0
    offpeak_rate: float = 6.0
    horizon_hours: int = 6
    max_parallel_jobs: int = 1
    node_limit: int = 50000


def default_appliances():
    return [Appliance("Refrigerator", 200, on=True, essential=True, flexible=False,
                      priority="High", ontology_id="Refrigerator_1"),
            Appliance("Air Conditioner", 1800, on=True, flexible=False, priority="High", requires_occupancy=True),
            Appliance("Washing Machine", 500, ontology_id="WashingMachine_Test"),
            Appliance("Dishwasher", 1200)]


def validate_inputs(appliances, scenario):
    if not 0 <= scenario.occupancy <= 20 or not np.isfinite(scenario.temperature_c):
        raise ValueError("Occupancy must be 0–20 and temperature must be finite.")
    if scenario.horizon_hours not in range(1, 13) or scenario.max_parallel_jobs not in range(1, 7):
        raise ValueError("Use a 1–12 hour horizon and 1–6 concurrent schedulable jobs.")
    if not 0 <= scenario.peak_start_hour < 24 or not 0 <= scenario.peak_end_hour < 24:
        raise ValueError("Tariff hours must be 0–23.")
    if any(not np.isfinite(v) or v < 0 for v in [scenario.peak_rate, scenario.offpeak_rate]):
        raise ValueError("Tariffs must be finite and non-negative.")
    if scenario.node_limit < 1 or not 1 <= len(appliances) <= 8:
        raise ValueError("Use 1–8 appliances and a positive search limit.")
    if len({a.name for a in appliances}) != len(appliances):
        raise ValueError("Appliance names must be unique.")
    for a in appliances:
        if any(type(getattr(a, field)) is not bool for field in ["requested", "on", "essential", "flexible", "requires_occupancy"]):
            raise ValueError("Appliance flags must be JSON true/false values.")
        if any(type(getattr(a, field)) is not int for field in ["duration_min", "earliest_min", "deadline_min"]):
            raise ValueError("Runtime, earliest start and deadline must be integer minutes.")
        if not a.name.strip() or not np.isfinite(a.power_w) or not 0 < a.power_w <= 20000:
            raise ValueError("Each appliance needs a name and a power rating in (0, 20000] W.")
        if a.priority not in ["Low", "Medium", "High"]:
            raise ValueError(f"Invalid priority for {a.name}.")
        if a.duration_min <= 0 or a.duration_min % 15 or a.earliest_min < 0 or a.earliest_min % 15 or a.deadline_min % 15:
            raise ValueError("Runtime, earliest start and deadline must use 15-minute steps.")
        if a.deadline_min <= a.earliest_min:
            raise ValueError(f"Deadline must follow earliest start for {a.name}.")


def peak_at(timestamp, scenario):
    hour = timestamp.hour + timestamp.minute / 60
    start, end = scenario.peak_start_hour, scenario.peak_end_hour
    if start == end:
        return False
    return start <= hour < end if start < end else hour >= start or hour < end


def rate_at(timestamp, scenario):
    return scenario.peak_rate if peak_at(timestamp, scenario) else scenario.offpeak_rate


def reason_appliance(appliance, scenario, origin, forecast_kw, threshold, knowledge):
    inferred = knowledge.infer(appliance)
    essential = "EssentialAppliance" in inferred["types"]
    flexible = appliance.flexible and "FlexibleAppliance" in inferred["types"]
    high_priority = "HighPriorityAppliance" in inferred["types"]
    trace = list(inferred["fired"])
    # Imported SWRL heads propose decisions. Ordered operational rules resolve
    # conflicts and add the tariff, occupancy, comfort and forecast conditions.
    if essential:
        action, reason, kind, rule = "Keep ON", "Essential appliance; protected regardless of tariff or forecast.", "protected", "R1-essential"
    elif scenario.occupancy == 0 and appliance.requires_occupancy:
        action, reason, kind, rule = "Turn OFF" if appliance.on else "Do not start", "Unoccupied home; this appliance requires occupancy.", "inactive", "R5-empty-home"
    elif appliance.name == "Air Conditioner" and scenario.occupancy > 0 and scenario.temperature_c >= 38:
        action, reason, kind, rule = "Keep ON", "Occupied home at or above 38°C; comfort overrides cost saving.", "protected", "R2-comfort"
    elif not appliance.requested and not appliance.on:
        action, reason, kind, rule = "Leave OFF", "Appliance was not requested.", "inactive", "R6-not-requested"
    elif high_priority or appliance.on:
        action, reason, kind, rule = "Run as scheduled", "High-priority or already running appliance; no automatic interruption or delay.", "protected", "R3-priority"
    elif flexible:
        high = forecast_kw >= threshold
        action, reason, kind, rule = "Find schedule", (
            "Flexible job; " + ("avoid the single forecast high-demand interval; " if high else "") +
            ("current tariff is peak; " if peak_at(origin, scenario) else "") + "search permitted start times for lowest estimated job cost."), "job", "R4-forecast-tariff"
    else:
        action, reason, kind, rule = "Run at earliest allowed time", "Non-flexible requested job keeps its earliest start.", "job", "R6-fixed-job"
    trace.append(rule)
    return {"Appliance": appliance.name, "Action": action, "Reason": reason, "Kind": kind,
            "Essential": essential, "Flexible": flexible, "Rule_trace": trace,
            "OWL_candidates": inferred["candidate_decisions"], "OWL_asserted_decisions": inferred["asserted_decisions"],
            "Ontology_types": inferred["types"]}


def plan_jobs(jobs, scenario, origin, target_end, forecast_high):
    """A* assignment search; admissible remaining-cost lower bound.

    Capacity counts only these schedulable jobs, not whole-house load. Forecast
    avoidance applies only to the known 15-minute target interval.
    """
    slots = scenario.horizon_hours * 4
    rates = np.array([rate_at(origin + pd.Timedelta(minutes=15 * i), scenario) for i in range(slots)])
    options = []
    for job, flexible in jobs:
        duration = job.duration_min // 15
        latest = min(slots, job.deadline_min // 15) - duration
        starts = list(range(job.earliest_min // 15, latest + 1))
        if not flexible:
            starts = starts[:1]
        choices = []
        for start in starts:
            begin, end = origin + pd.Timedelta(minutes=15 * start), origin + pd.Timedelta(minutes=15 * (start + duration))
            overlaps_forecast = begin < target_end and end > target_end - pd.Timedelta(minutes=15)
            if flexible and forecast_high and overlaps_forecast:
                continue
            cost = float((job.power_w / 1000) * .25 * rates[start:start + duration].sum())
            choices.append((start, duration, cost))
        if not choices:
            return {"status": "infeasible", "reason": f"No allowed start for {job.name}; check deadline and forecast avoidance.",
                    "schedule": [], "estimated_cost": None, "baseline_cost": None, "estimated_saving": None, "nodes": 0}
        options.append(choices)
    if not jobs:
        return {"status": "optimal", "reason": "No schedulable jobs requested.", "schedule": [],
                "estimated_cost": 0., "baseline_cost": 0., "estimated_saving": 0., "nodes": 0}
    # A chronological DFS supplies a feasible earliest-start comparison under
    # exactly the same constraints. It is not claimed to be the actual old bill.
    baseline_nodes = 0
    baseline_limit = False
    def first_feasible(index, loads, starts):
        nonlocal baseline_nodes, baseline_limit
        if baseline_nodes >= scenario.node_limit:
            baseline_limit = True
            return None
        baseline_nodes += 1
        if index == len(jobs):
            return starts
        for start, duration, cost in options[index]:
            if max(loads[start:start + duration], default=0) >= scenario.max_parallel_jobs:
                continue
            updated = list(loads)
            for i in range(start, start + duration):
                updated[i] += 1
            result = first_feasible(index + 1, updated, starts + [(start, duration, cost)])
            if result is not None:
                return result
        return None
    baseline = first_feasible(0, [0] * slots, [])
    if baseline is None:
        return {"status": "search_limit" if baseline_limit else "infeasible",
                "reason": "No feasible schedule found before the search limit." if baseline_limit else "Requested jobs cannot all meet their windows and concurrency limit.",
                "schedule": [], "estimated_cost": None, "baseline_cost": None, "estimated_saving": None, "nodes": 0}
    minimum = [min(x[2] for x in choices) for choices in options]
    lower = [sum(minimum[i:]) for i in range(len(jobs) + 1)]
    queue, counter = [], count()
    heapq.heappush(queue, (lower[0], 0., next(counter), 0, (0,) * slots, []))
    best = {(0, (0,) * slots): 0.}
    chosen, status, explored = baseline, "feasible_search_limit", 0
    while queue and explored < scenario.node_limit:
        _, cost_so_far, _, index, loads, path = heapq.heappop(queue)
        if cost_so_far > best.get((index, loads), float("inf")) + 1e-10:
            continue
        explored += 1
        if index == len(jobs):
            chosen, status = path, "optimal"
            break
        for start, duration, cost in options[index]:
            if max(loads[start:start + duration], default=0) >= scenario.max_parallel_jobs:
                continue
            updated = list(loads)
            for i in range(start, start + duration):
                updated[i] += 1
            new_cost, state = cost_so_far + cost, (index + 1, tuple(updated))
            if new_cost + 1e-10 < best.get(state, float("inf")):
                best[state] = new_cost
                heapq.heappush(queue, (new_cost + lower[index + 1], new_cost, next(counter),
                                       index + 1, state[1], path + [(start, duration, cost)]))
    rows = []
    for (job, _), (start, duration, cost) in zip(jobs, chosen):
        rows.append({"Appliance": job.name, "Start": str(origin + pd.Timedelta(minutes=15 * start)),
                     "End": str(origin + pd.Timedelta(minutes=15 * (start + duration))),
                     "Power_W": job.power_w, "Energy_kWh": job.power_w / 1000 * duration * .25,
                     "Estimated_cost": cost})
    total, baseline_cost = sum(p[2] for p in chosen), sum(p[2] for p in baseline)
    return {"status": status, "reason": "Minimum cost under declared constraints." if status == "optimal" else "Feasible DFS schedule retained; A* reached its node limit, so optimality is unproven.",
            "schedule": rows, "estimated_cost": total, "baseline_cost": baseline_cost,
            "estimated_saving": baseline_cost - total, "nodes": explored}


def optimize_decisions(forecast, appliances, scenario, knowledge):
    validate_inputs(appliances, scenario)
    origin, target = pd.Timestamp(forecast["origin"]), pd.Timestamp(forecast["target_end"])
    value, threshold = float(forecast["forecast_kw"]), float(forecast["threshold_kw"])
    if not np.isfinite([value, threshold]).all() or value < 0 or threshold <= 0:
        raise ValueError("Forecast and training threshold must be finite valid power values.")
    if target - origin != pd.Timedelta(hours=1):
        raise ValueError("Expected a one-hour-ahead forecast target.")
    rows = [reason_appliance(a, scenario, origin, value, threshold, knowledge) for a in appliances]
    jobs = [(a, row["Flexible"]) for a, row in zip(appliances, rows) if row["Kind"] == "job"]
    plan = plan_jobs(jobs, scenario, origin, target, value >= threshold)
    for row in rows:
        if row["Kind"] != "job":
            continue
        scheduled = next((s for s in plan["schedule"] if s["Appliance"] == row["Appliance"]), None)
        if scheduled is None:
            row["Action"] = "No feasible schedule — review inputs"
            row["Suggested_start"] = None
        else:
            row["Suggested_start"] = scheduled["Start"]
            row["Action"] = "Delay" if pd.Timestamp(scheduled["Start"]) > origin else "Run now"
    return {"forecast": forecast, "scenario": asdict(scenario), "appliances": [asdict(a) for a in appliances],
            "recommendations": rows, "plan": plan, "ontology_sha256": knowledge.digest,
            "provenance": "Electricity history is recorded data; household settings are user-supplied scenarios. Savings concern requested schedulable jobs only, not measured whole-house bills."}

## 4. Connect the learned models
Frozen LSTM summaries represent past patterns. A classical ML regressor then
predicts power using those summaries plus the Task 7 context features.
Gradient Boosting and Random Forest reuse the algorithm families from Task 6;
their new weights must be learned on this new representation. Selection uses
validation RMSE only. Training examples are sampled evenly from the training
period. The LSTM itself is not trained by this new ML head.

In [ ]:
class ForecastBridge:
    """Frozen Task 7 LSTM features + validation-selected Task 6 algorithm family."""
    def __init__(self, artifact_dir, output_root, train_cap=20000, validation_cap=5000, force_fit=False):
        import joblib
        from tensorflow import keras
        from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
        self.folder, self.output_root = Path(artifact_dir), Path(output_root)
        self.manifest = json.loads((self.folder / "manifest.json").read_text(encoding="utf-8"))
        if self.manifest["run_mode"] != "full" or self.manifest["horizon_steps"] != 4 or self.manifest["sequence_length"] != 96:
            raise ValueError("Use a full updated Task 7 run with 96 steps and a four-step horizon.")
        self.network = keras.models.load_model(self.folder / "task7_FINAL_lstm.keras", compile=False)
        self.scaler = joblib.load(self.folder / "power_scaler.joblib")
        self.mean, self.scale = float(self.scaler.mean_[0]), float(self.scaler.scale_[0])
        with np.load(self.folder / "prepared_arrays.npz", allow_pickle=False) as archive:
            self.arrays = {key: archive[key] for key in archive.files}
        self.history = self.arrays["history_values"]
        self.context = self.arrays["context_values"]
        self.timestamps = pd.DatetimeIndex(pd.to_datetime(self.arrays["timestamps_ns"]))
        self.y = self.arrays["target_scaled"].reshape(-1) * self.scale + self.mean
        self.train_ids = self._sample(self.arrays["train_indices"], train_cap)
        self.validation_ids = self._sample(self.arrays["validation_indices"], validation_cap)
        self.test_ids = self.arrays["test_indices"]
        if not len(self.train_ids) or not len(self.validation_ids) or not len(self.test_ids):
            raise ValueError("All temporal partitions must be non-empty.")
        assert self.train_ids.max() < self.validation_ids.min() - 4
        assert self.validation_ids.max() < self.test_ids.min() - 4
        self.encoder = keras.Model(self.network.inputs, self.network.get_layer("lstm_summary").output)
        self.input_order = [tensor.name.split(":")[0] for tensor in self.network.inputs]
        self.feature_names = [f"LSTM_pattern_{i+1:02d}" for i in range(int(self.encoder.output_shape[-1]))] + ["Context: " + n for n in self.manifest["context_features"]]
        self.signature = hashlib.sha256(json.dumps({"model": file_hash(self.folder / "task7_FINAL_lstm.keras"),
            "arrays": file_hash(self.folder / "prepared_arrays.npz"), "manifest": file_hash(self.folder / "manifest.json"),
            "scaler": file_hash(self.folder / "power_scaler.joblib"),
            "train_cap": train_cap, "validation_cap": validation_cap, "schema": 1}, sort_keys=True).encode()).hexdigest()
        self.cache = self.output_root / "model_cache" / self.signature[:16]
        self.cache.mkdir(parents=True, exist_ok=True)
        cache_file = self.cache / "hybrid_forecaster.joblib"
        if cache_file.is_file() and not force_fit:
            saved = joblib.load(cache_file)
            if saved["signature"] != self.signature:
                raise ValueError("Hybrid-model cache does not match this Task 7 run.")
            self.regressor, self.selected_name = saved["model"], saved["selected_name"]
            self.validation_results = pd.DataFrame(saved["validation_results"])
            print("Reusing cached ML prediction stage:", self.selected_name)
        else:
            print(f"Extracting frozen LSTM patterns: {len(self.train_ids)} training / {len(self.validation_ids)} validation examples...")
            train_z, val_z = self.features(self.train_ids), self.features(self.validation_ids)
            candidates = {"LSTM + Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=.05, random_state=42),
                          "LSTM + Random Forest": RandomForestRegressor(n_estimators=60, max_depth=12, min_samples_leaf=4, random_state=42, n_jobs=2)}
            records = []
            for name, model in candidates.items():
                print("Fitting ML stage:", name, flush=True)
                model.fit(train_z, self.y[self.train_ids])
                prediction = np.maximum(model.predict(val_z), 0)
                records.append({"Model": name, **self.metrics(self.y[self.validation_ids], prediction)})
            self.validation_results = pd.DataFrame(records).sort_values("RMSE_kW").reset_index(drop=True)
            self.selected_name = self.validation_results.iloc[0]["Model"]
            self.regressor = candidates[self.selected_name]
            joblib.dump({"signature": self.signature, "model": self.regressor, "selected_name": self.selected_name,
                         "validation_results": self.validation_results.to_dict("records")}, cache_file)
        self.validation_results.to_csv(self.cache / "validation_comparison.csv", index=False)
        self.threshold = float(self.manifest["peak_threshold_kw"])

    @staticmethod
    def _sample(ids, cap):
        ids = np.asarray(ids, dtype=np.int64)
        return ids[np.unique(np.linspace(0, len(ids) - 1, min(len(ids), cap), dtype=int))]

    @staticmethod
    def metrics(actual, predicted):
        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
        return {"MAE_kW": float(mean_absolute_error(actual, predicted)),
                "RMSE_kW": float(np.sqrt(mean_squared_error(actual, predicted))), "R2": float(r2_score(actual, predicted))}

    def inputs(self, ids):
        ids = np.asarray(ids, dtype=np.int64)
        positions = ids[:, None] - 4 - 96 + 1 + np.arange(96)[None, :]
        if positions.min() < 0:
            raise ValueError("Not enough history for this forecast.")
        result = {"history": np.asarray(self.history[positions], dtype=np.float32),
                  "context": np.asarray(self.context[ids], dtype=np.float32)}
        if not all(np.isfinite(v).all() for v in result.values()):
            raise ValueError("A requested forecast contains unavailable input features.")
        return result

    def features(self, ids, batch_size=256):
        batches = []
        for offset in range(0, len(ids), batch_size):
            x = self.inputs(ids[offset:offset+batch_size])
            latent = np.asarray(self.encoder([x[key] for key in self.input_order], training=False))
            batches.append(np.concatenate([latent, x["context"]], axis=1))
        return np.concatenate(batches)

    def predict(self, target_index):
        target_index = int(target_index)
        if target_index not in self.test_ids:
            raise ValueError("Dashboard examples must come from the held-out test period.")
        z = self.features(np.array([target_index]))
        raw = float(self.regressor.predict(z)[0])
        return {"origin": str(self.timestamps[target_index - 4]), "target_end": str(self.timestamps[target_index]),
                "target_interval_start": str(self.timestamps[target_index] - pd.Timedelta(minutes=15)),
                "forecast_kw": max(raw, 0.), "raw_forecast_kw": raw, "threshold_kw": self.threshold,
                "model": self.selected_name, "target_index": target_index,
                "units": "kW, average over the target 15-minute interval"}

    def evaluate(self):
        from tensorflow import keras
        ids = self.test_ids
        hybrid = np.maximum(self.regressor.predict(self.features(ids)), 0)
        saved = pd.read_csv(self.folder / "test_predictions.csv", parse_dates=["Target_interval_end"])
        matched = saved.set_index("Target_interval_end").loc[self.timestamps[ids]]
        np.testing.assert_allclose(self.y[ids], matched["Actual_kW"], atol=1e-5)
        predictions = {self.selected_name: hybrid,
            "Task 7 LSTM": matched[self.manifest["selection"]["selected_lstm"] + "_kW"].to_numpy(),
            "Basic neural network": matched["basic_mlp_kW"].to_numpy(), "Persistence": matched["Persistence_kW"].to_numpy()}
        records = []
        for segment, mask in {"All test": np.ones(len(ids), dtype=bool), "High demand": self.y[ids] >= self.threshold}.items():
            for name, values in predictions.items():
                metrics = self.metrics(self.y[ids][mask], values[mask]) if mask.sum() >= 2 else {"MAE_kW": None, "RMSE_kW": None, "R2": None}
                records.append({"Model": name, "Segment": segment, "Samples": int(mask.sum()), **metrics})
        result = pd.DataFrame(records)
        result.to_csv(self.cache / "full_test_comparison.csv", index=False)
        return result

    def explain(self, target_index):
        import shap
        z = self.features(np.array([target_index]))
        explainer = shap.TreeExplainer(self.regressor)
        values = np.asarray(explainer.shap_values(z)).reshape(-1)
        base = float(np.asarray(explainer.expected_value).reshape(-1)[0])
        raw = float(self.regressor.predict(z)[0])
        np.testing.assert_allclose(base + values.sum(), raw, rtol=1e-5, atol=1e-5)
        return {"values": values, "base": base, "raw_prediction": raw, "data": z[0], "features": self.feature_names}

    def activation(self, target_index):
        from tensorflow import keras
        x = self.inputs(np.array([target_index]))
        view = keras.Model(self.network.inputs, self.network.get_layer("lstm_temporal_features").output)
        return np.asarray(view([x[k] for k in self.input_order], training=False))[0]


def find_task7_folder(explicit="", roots=None):
    required = ["manifest.json", "task7_FINAL_lstm.keras", "task7_basic_mlp.keras", "power_scaler.joblib",
                "prepared_arrays.npz", "test_predictions.csv"]
    def complete(folder):
        try:
            return all((folder / name).is_file() for name in required) and json.loads((folder / "manifest.json").read_text(encoding="utf-8"))["run_mode"] == "full"
        except (ValueError, KeyError):
            return False
    if explicit:
        folder = Path(explicit).expanduser()
        if not complete(folder):
            raise ValueError(f"Incomplete full Task 7 output folder: {folder}")
        return folder.resolve()
    folders = set()
    for root in roots or [Path.cwd()]:
        root = Path(root)
        if complete(root):
            folders.add(root.resolve())
        for manifest in (root / "task7_FINAL_outputs").glob("*/manifest.json"):
            if complete(manifest.parent):
                folders.add(manifest.parent.resolve())
    return max(folders, key=lambda folder: (folder / "manifest.json").stat().st_mtime) if folders else None

In [ ]:
knowledge = OntologyKnowledge(ONTOLOGY_XML)
print("Ontology classes:", len(knowledge.classes))
print("Imported enabled SWRL rules:", [rule["id"] for rule in knowledge.rules])
print("Original asserted examples:", knowledge.asserted_decisions)
print("Ontology SHA-256:", knowledge.digest)

def prepare_task7_results():
    roots = [Path.cwd()]
    for key in ["TASK7_COMPLETED_OUTPUT_DIR", "ARTIFACT_DIR", "OUTPUT_DIR"]:
        value = globals().get(key)
        if value:
            roots.append(Path(value))
    folder = find_task7_folder(TASK7_FOLDER, roots)
    if folder is not None:
        print("Reusing saved Task 7 models:", folder)
        return folder
    if not BUILD_TASK7_IF_MISSING:
        raise FileNotFoundError("No completed updated Task 7 folder is available in this runtime. Set TASK7_FOLDER, or enable BUILD_TASK7_IF_MISSING.")
    print("Fresh runtime: building Task 7 models once from the real UCI dataset. This includes model training.", flush=True)
    print("SHAP and the dashboard run after model preparation. No ZIP transfer is required.", flush=True)
    # Keras temporary model files stay within this writable project directory.
    old_tempdir = tempfile.tempdir
    tempdir = (TASK9_OUTPUT_DIR / "runtime_temp").resolve()
    tempdir.mkdir(parents=True, exist_ok=True)
    namespace = {"__name__": "task7_bootstrap"}
    try:
        tempfile.tempdir = str(tempdir)
        exec(compile(TASK7_SOURCE_TEXT, "embedded_task7_unchanged.py", "exec"), namespace)
        folder = Path(namespace["OUTPUT_DIR"]).resolve()
    finally:
        tempfile.tempdir = old_tempdir
    return find_task7_folder(str(folder))

task7_folder = prepare_task7_results()
bridge = ForecastBridge(task7_folder, TASK9_OUTPUT_DIR, TRAIN_EXAMPLES, VALIDATION_EXAMPLES, FORCE_REFIT_ML)
print("Task 9 selected ML stage:", bridge.selected_name)
display(bridge.validation_results.round(4))
performance = bridge.evaluate()
display(performance.round(4))
print("The hybrid is not assumed to beat the original LSTM or basic neural network; compare the generated metrics.")

## 5. End-to-end consistency checks
These are controlled decision scenarios, not new measured electricity results.
They check imported rules, protected appliances, occupancy, forecast feedback,
scheduling feasibility and error handling. Task 10's broader unseen-data,
fairness and deployment evaluation remains separate.

In [ ]:
from dataclasses import replace
check_forecast = bridge.predict(int(bridge.test_ids[0]))
scenarios = []
fridge = Appliance("Refrigerator", 200, on=True, essential=False, flexible=True, ontology_id="Refrigerator_1")
essential_check = optimize_decisions(check_forecast, [fridge], Scenario(occupancy=0), knowledge)
assert essential_check["recommendations"][0]["Action"] == "Keep ON"
scenarios.append({"Check": "Ontology protects refrigerator even if editable essential flag is false", "Passed": True})
ac = Appliance("Air Conditioner", 1800, on=True, flexible=False, priority="High", requires_occupancy=True)
occupied_ac = optimize_decisions(check_forecast, [ac], Scenario(temperature_c=40, occupancy=3), knowledge)
empty_ac = optimize_decisions(check_forecast, [ac], Scenario(temperature_c=40, occupancy=0), knowledge)
assert occupied_ac["recommendations"][0]["Action"] == "Keep ON"
assert empty_ac["recommendations"][0]["Action"] == "Turn OFF"
scenarios.append({"Check": "Comfort protects occupied AC; empty-home AC can turn off", "Passed": True})
high_forecast = dict(check_forecast, forecast_kw=bridge.threshold + 1)
wash = Appliance("Washing Machine", 500, ontology_id="WashingMachine_Test")
high_result = optimize_decisions(high_forecast, [wash], Scenario(), knowledge)
assert high_result["plan"]["status"] == "optimal"
for slot in high_result["plan"]["schedule"]:
    assert not (pd.Timestamp(slot["Start"]) < pd.Timestamp(high_forecast["target_end"]) and
                pd.Timestamp(slot["End"]) > pd.Timestamp(high_forecast["target_end"]) - pd.Timedelta(minutes=15))
scenarios.append({"Check": "Forecast high demand prevents flexible job overlap with the one known target interval", "Passed": True})
impossible = optimize_decisions(high_forecast, [replace(wash, deadline_min=30, duration_min=60)], Scenario(), knowledge)
assert impossible["plan"]["status"] == "infeasible" and impossible["plan"]["estimated_saving"] is None
scenarios.append({"Check": "Impossible deadline is reported without claimed savings", "Passed": True})
checks = pd.DataFrame(scenarios)
checks.to_csv(TASK9_OUTPUT_DIR / "decision_consistency_checks.csv", index=False)
display(checks)

## 6. Dashboard
Select a historical case and edit the household scenario. In each appliance's
expandable settings, change requested jobs, power ratings,
durations, priorities, earliest starts and deadlines using the controls.
Use multiples of 15 minutes for scheduling values.

The forecast means average kW over one 15-minute interval ending one hour
after the selected origin. High-demand avoidance applies only to that interval;
no full-day demand profile is fabricated. Later scheduling choices use the
supplied tariffs and job constraints. Equal peak-start/end hours mean no peak
period. The concurrency control limits schedulable jobs, not total home load.

SHAP TreeExplainer explains the classical ML head in its learned-feature space.
Its reconstruction is checked against the raw prediction before non-negative
clipping. LSTM activation heatmaps inspect internal responses. This is not an
exact sensor-level explanation of the entire composed pipeline or a causal
explanation. These uses extend the SHAP/activation approach from Task 8.
[SHAP TreeExplainer reference](https://shap.readthedocs.io/en/stable/generated/shap.TreeExplainer.html).

In [ ]:
"""Interactive in-notebook controls and Task 9 result views."""
from datetime import datetime, timezone
from html import escape
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, clear_output, display

if "Appliance" not in globals():
    from task9_core import Appliance, Scenario, default_appliances, optimize_decisions


def render_optimizer_result(result, bridge, explanation=None):
    forecast, plan = result["forecast"], result["plan"]
    high = forecast["forecast_kw"] >= forecast["threshold_kw"]
    saving = "Unavailable" if plan["estimated_saving"] is None else f"₹{plan['estimated_saving']:.2f}"
    cards = [("FORECAST AVERAGE POWER", f"{forecast['forecast_kw']:.3f} kW"),
             ("FORECAST DEMAND", "High" if high else "Normal"),
             ("SCHEDULE STATUS", plan["status"].replace("_", " ")),
             ("ESTIMATED JOB SAVING", saving)]
    markup = '<div style="display:flex;flex-wrap:wrap;gap:12px;font-family:Arial,sans-serif">'
    for label, value in cards:
        markup += f'<div style="background:#edf5fa;border-left:4px solid #176b87;padding:15px;min-width:185px;border-radius:8px"><small>{escape(label)}</small><br><strong style="font-size:23px">{escape(value)}</strong></div>'
    markup += '</div>'
    display(HTML(markup))
    display(HTML(f"<p><b>Origin:</b> {escape(forecast['origin'])} &nbsp; <b>Predicted interval:</b> "
                 f"{escape(forecast['target_interval_start'])} to {escape(forecast['target_end'])}<br>"
                 f"<b>Model:</b> {escape(forecast['model'])}. High demand starts at {forecast['threshold_kw']:.3f} kW.</p>"))
    display(HTML("<p>Historical electricity case with editable household scenario settings. "
                 "Savings cover the requested schedulable jobs versus a feasible earliest-start schedule; they are estimates, not a measured household bill.</p>"))
    rows = [{"Appliance": r["Appliance"], "Recommendation": r["Action"],
             "Suggested start": r.get("Suggested_start") or "Protected / inactive",
             "Reason": r["Reason"]} for r in result["recommendations"]]
    display(pd.DataFrame(rows))
    display(HTML(f"<p><b>Search:</b> {escape(plan['reason'])} Explored {plan['nodes']} A* states. "
                 "The concurrency setting counts scheduled jobs, not the entire household.</p>"))
    input_values = bridge.inputs(np.array([forecast["target_index"]]))
    history_names = bridge.manifest["history_features"]
    power_channel = next(i for i, name in enumerate(history_names) if "global_active_power" in name.lower())
    history_kw = input_values["history"][0, :, power_channel] * bridge.scale + bridge.mean
    origin, target = pd.Timestamp(forecast["origin"]), pd.Timestamp(forecast["target_end"])
    history_times = pd.date_range(end=origin, periods=len(history_kw), freq="15min")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history_times, history_kw, color="#176b87", label="Historical model inputs")
    axes[0].scatter([target], [forecast["forecast_kw"]], color="#d86526", s=65, label="One-hour-ahead target")
    axes[0].axvspan(target-pd.Timedelta(minutes=15), target, alpha=.15, color="#d86526")
    axes[0].set(title="Recent history and forecast target", ylabel="Average power (kW)")
    axes[0].legend(fontsize=8)
    axes[0].tick_params(axis="x", rotation=25)
    if plan["estimated_cost"] is not None:
        axes[1].bar(["Feasible earliest start", "A* schedule"], [plan["baseline_cost"], plan["estimated_cost"]], color=["#94a3b8", "#31a6a0"])
        axes[1].set(title="Estimated cost of scheduled jobs", ylabel="Scenario cost (₹)")
    else:
        axes[1].text(.5, .5, "No feasible cost estimate\nReview scheduling constraints", ha="center", va="center", transform=axes[1].transAxes)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    if plan["schedule"]:
        schedule = pd.DataFrame(plan["schedule"])
        display(schedule)
        fig, ax = plt.subplots(figsize=(12, 3.5))
        for i, row in schedule.iterrows():
            start = (pd.Timestamp(row["Start"]) - origin).total_seconds() / 3600
            duration = (pd.Timestamp(row["End"]) - pd.Timestamp(row["Start"])).total_seconds() / 3600
            ax.barh(i, duration, left=start, color="#31a6a0", height=.55)
        ax.set_yticks(range(len(schedule)), schedule["Appliance"])
        tick_values = np.arange(result["scenario"]["horizon_hours"] + 1)
        ax.set_xticks(tick_values, [(origin + pd.Timedelta(hours=int(h))).strftime("%H:%M") for h in tick_values])
        if high:
            ax.axvspan(.75, 1., alpha=.15, color="red", label="Single forecast high-demand interval")
            ax.legend(fontsize=8)
        ax.set(xlabel=f"Planning clock from {origin.date()}", title="Requested-job schedule")
        fig.tight_layout()
        plt.show()
        plt.close(fig)
    with_details = "<details><summary><b>Why were these decisions made? — ontology and operational rule trace</b></summary>"
    for row in result["recommendations"]:
        with_details += f"<p><b>{escape(row['Appliance'])}</b>: {escape(' → '.join(row['Rule_trace']))}<br>"
        with_details += f"OWL candidate decisions: {escape(', '.join(row['OWL_candidates']) or 'None')}. "
        with_details += f"Final decision: {escape(row['Action'])}. {escape(row['Reason'])}</p>"
    display(HTML(with_details + "</details>"))
    if explanation is not None:
        import shap
        display(HTML("<h4>SHAP: why the ML prediction stage produced this forecast</h4>"
                     "<p>Features include learned LSTM patterns and context inputs. These SHAP values explain the ML stage, "
                     "not separate causal effects of raw sensors. LSTM activations are shown below.</p>"))
        obj = shap.Explanation(values=explanation["values"], base_values=explanation["base"],
                               feature_names=explanation["features"])
        shap.plots.waterfall(obj, max_display=12, show=False)
        fig = plt.gcf()
        fig.set_size_inches(12, 7)
        plt.title(f"Raw ML forecast: {explanation['raw_prediction']:.3f} kW; final clipped forecast: {forecast['forecast_kw']:.3f} kW", fontsize=10)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
        activation = bridge.activation(forecast["target_index"])
        fig, ax = plt.subplots(figsize=(12, 4))
        image = ax.imshow(activation.T, aspect="auto", cmap="RdBu_r")
        ticks = np.linspace(0, activation.shape[0]-1, 7, dtype=int)
        ax.set_xticks(ticks, [f"{(activation.shape[0]-1-t)*.25:g}" for t in ticks])
        ax.set(title="LSTM temporal activations — internal response, not attention weights", xlabel="Hours before forecast origin", ylabel="LSTM unit")
        fig.colorbar(image, ax=ax, label="Hidden-state activation")
        fig.tight_layout()
        plt.show()
        plt.close(fig)


def build_optimizer_dashboard(bridge, knowledge, output_dir, performance=None):
    from dataclasses import asdict
    # Defined in task9_core, available as imports locally or as earlier notebook definitions.
    ids = bridge._sample(bridge.test_ids, 64)
    evening = bridge.test_ids[bridge.timestamps[bridge.test_ids - 4].hour == 18]
    if len(evening):
        ids = np.unique(np.r_[ids, evening[0]])
    case = widgets.Dropdown(options=[(str(bridge.timestamps[int(i)-4]), int(i)) for i in ids],
                            value=int(evening[0] if len(evening) else ids[0]), description="History case:", layout=widgets.Layout(width="470px"))
    occupancy = widgets.IntSlider(value=3, min=0, max=10, description="People home:")
    temperature = widgets.FloatSlider(value=35, min=0, max=50, step=.5, description="Temp (°C):")
    peak_rate = widgets.BoundedFloatText(value=9, min=0, max=1000, description="Peak ₹/kWh:")
    offpeak_rate = widgets.BoundedFloatText(value=6, min=0, max=1000, description="Other ₹/kWh:")
    peak_start = widgets.IntSlider(value=18, min=0, max=23, description="Peak starts:")
    peak_end = widgets.IntSlider(value=20, min=0, max=23, description="Peak ends:")
    horizon = widgets.Dropdown(options=[4, 6, 8, 12], value=6, description="Plan hours:")
    concurrency = widgets.IntSlider(value=1, min=1, max=3, description="Parallel jobs:")
    appliance_forms, appliance_sections = [], []
    for appliance in default_appliances():
        initial = asdict(appliance)
        locked = "EssentialAppliance" in knowledge.infer(appliance)["types"]
        options = {"requested": widgets.Checkbox(value=appliance.requested, description="Requested"),
            "on": widgets.Checkbox(value=appliance.on, description="Already running"),
            "essential": widgets.Checkbox(value=appliance.essential or locked, description="Essential", disabled=locked),
            "flexible": widgets.Checkbox(value=appliance.flexible, description="Allow rescheduling"),
            "requires_occupancy": widgets.Checkbox(value=appliance.requires_occupancy, description="Needs people home"),
            "priority": widgets.Dropdown(value=appliance.priority, options=["Low", "Medium", "High"], description="Priority:"),
            "power_w": widgets.BoundedFloatText(value=appliance.power_w, min=1, max=20000, description="Power (W):"),
            "duration_min": widgets.BoundedIntText(value=appliance.duration_min, min=15, max=720, step=15, description="Runtime (min):"),
            "earliest_min": widgets.BoundedIntText(value=appliance.earliest_min, min=0, max=720, step=15, description="Earliest (min):"),
            "deadline_min": widgets.BoundedIntText(value=appliance.deadline_min, min=15, max=720, step=15, description="Finish by (min):")}
        for control in options.values():
            control.layout.width = "245px"
            if hasattr(control, "style"):
                control.style.description_width = "105px"
        grid = widgets.GridBox(list(options.values()), layout=widgets.Layout(
            grid_template_columns="repeat(3, minmax(245px, 1fr))", grid_gap="6px", width="100%", overflow="auto"))
        appliance_sections.append(grid)
        appliance_forms.append({"initial": initial, "controls": options})
    advanced = widgets.Accordion(children=appliance_sections)
    for i, appliance in enumerate(default_appliances()):
        advanced.set_title(i, appliance.name)
    appliance_help = widgets.HTML("<p>Expand an appliance to change its settings. Earliest start and finish deadline are minutes "
        "after the selected origin, in 15-minute steps. Defaults are scenario examples. Essential appliances remain protected.</p>")
    explain = widgets.Checkbox(value=True, description="Show SHAP and LSTM activations", indent=False)
    button = widgets.Button(description="Run optimizer", button_style="success", icon="play", layout=widgets.Layout(width="180px"))
    status, result_output = widgets.HTML(), widgets.Output()
    state = {"last_result": None, "last_path": None}

    def run(_=None):
        button.disabled = True
        status.value = "<p>Calculating forecast and schedule…</p>"
        try:
            appliance_values = [Appliance(**dict(form["initial"], **{key: control.value for key, control in form["controls"].items()}))
                                for form in appliance_forms]
            scenario = Scenario(occupancy=occupancy.value, temperature_c=temperature.value,
                peak_start_hour=peak_start.value, peak_end_hour=peak_end.value, peak_rate=peak_rate.value,
                offpeak_rate=offpeak_rate.value, horizon_hours=horizon.value, max_parallel_jobs=concurrency.value)
            forecast = bridge.predict(case.value)
            result = optimize_decisions(forecast, appliance_values, scenario, knowledge)
            explanation = bridge.explain(case.value) if explain.value else None
            folder = Path(output_dir) / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
            folder.mkdir(parents=True, exist_ok=False)
            report = dict(result, task7_run_id=bridge.manifest.get("run_id"), hybrid_signature=bridge.signature)
            if explanation is not None:
                report["ml_shap"] = {"base_kw": explanation["base"], "raw_forecast_kw": explanation["raw_prediction"],
                                     "features": explanation["features"], "values_kw": explanation["values"].tolist()}
            (folder / "optimizer_result.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
            pd.DataFrame(result["recommendations"]).to_csv(folder / "recommendations.csv", index=False)
            pd.DataFrame(result["plan"]["schedule"]).to_csv(folder / "schedule.csv", index=False)
            with result_output:
                clear_output(wait=True)
                render_optimizer_result(result, bridge, explanation)
                if performance is not None:
                    display(HTML("<details><summary><b>Forecasting performance on the full saved test period</b></summary></details>"))
                    display(performance.round(4))
                print("Saved this scenario:", folder.resolve())
            state["last_result"], state["last_path"] = result, folder
            status.value = "<p style='color:#126b46'><b>Optimizer complete.</b> Change settings and run again to compare scenarios.</p>"
        except Exception as error:
            status.value = f"<p style='color:#b42318'><b>Inputs need attention:</b> {escape(str(error))}</p>"
            with result_output:
                clear_output(wait=True)
                print("No new result was produced. Review the message above.")
        finally:
            button.disabled = False

    button.on_click(run)
    header = widgets.HTML("<h2 style='color:#176b87'>Smart Electricity Consumption Optimizer</h2>"
        "<p><b>Task 9:</b> LSTM patterns → ML forecast → ontology/rules → A* schedule → explanation.</p>"
        "<p>Choose a recorded electricity case and enter a household scenario. Tariffs and appliance settings below are editable examples. "
        "The forecast follows the selected electricity history; household settings affect decisions and estimated costs. "
        "This dashboard provides recommendations; it does not switch hardware.</p>")
    controls = widgets.VBox([header, case,
        widgets.HBox([occupancy, temperature]), widgets.HBox([peak_rate, offpeak_rate]),
        widgets.HBox([peak_start, peak_end]), widgets.HBox([horizon, concurrency]), appliance_help, advanced, explain,
        widgets.HBox([button, status]), result_output], layout=widgets.Layout(width="100%"))
    return {"widget": controls, "run": run, "state": state,
            "controls": {"case": case, "occupancy": occupancy, "temperature": temperature,
                         "appliances": appliance_forms, "explain": explain, "peak_rate": peak_rate,
                         "offpeak_rate": offpeak_rate, "horizon": horizon, "concurrency": concurrency}}

In [ ]:
dashboard = build_optimizer_dashboard(bridge, knowledge, TASK9_OUTPUT_DIR / "scenarios", performance)
display(dashboard["widget"])
dashboard["run"]()  # Initial result; subsequent clicks perform inference and scheduling, not training.

## Deliverables and interpretation
Every successful dashboard run writes a JSON report plus recommendation and
schedule CSVs under `task9_outputs/scenarios`. The hybrid model, validation
comparison and full-test comparison are cached separately. Save the executed
notebook for submission. No model accuracy improvement or measured bill saving
is claimed without evidence. Recommendations preserve essential/comfort rules;
scheduling is optimal only when the search status says `optimal`, under the
declared scenario constraints. Existing Tasks 1–8 files remain separate.